In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.stats import ks_2samp

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, roc_auc_score, silhouette_score
)

BG, GRID, TXT = '#0b1730', '#2a3655', '#dfe7f5'
AZUL, VERMELHO = '#3b82f6', '#d64045'
PALETA = {'Aprovado': AZUL, 'Reprovado': VERMELHO}
HUE_ORDER = ['Aprovado', 'Reprovado']

def novo_grafico(figsize=(12, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    return fig, ax

def estilizar(ax, titulo, xlabel='', ylabel='', grid_eixo='y'):
    ax.set_title(titulo, fontsize=16, color='white', weight='bold', pad=15)
    ax.set_xlabel(xlabel, color=TXT, fontsize=12)
    ax.set_ylabel(ylabel, color=TXT, fontsize=12)
    ax.tick_params(axis='both', colors=TXT, labelsize=11)
    ax.grid(False)
    ax.grid(True, axis=grid_eixo, linestyle='-', alpha=0.3, color=GRID)
    ax.set_axisbelow(True)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']:
        ax.spines[s].set_color(GRID)
    leg = ax.get_legend()
    if leg is not None:
        leg.get_title().set_color('white')
        for t in leg.get_texts():
            t.set_color('white')
        leg.get_frame().set_facecolor(BG)
        leg.get_frame().set_edgecolor(GRID)

def rotular_barras(ax, horizontal=False):
    for p in ax.patches:
        v = p.get_width() if horizontal else p.get_height()
        if v and v > 0:
            if horizontal:
                ax.text(v + 0.5, p.get_y() + p.get_height()/2, f'{int(v)}',
                        va='center', ha='left', color='white', fontsize=9)
            else:
                ax.text(p.get_x() + p.get_width()/2, v + 0.5, f'{int(v)}',
                        ha='center', va='bottom', color='white', fontsize=9)

print('Ambiente pronto.')

In [ ]:
ARQUIVO = 'database_recruiter.csv'
df = pd.read_csv(ARQUIVO, encoding='utf-8-sig')

# Ajuste de tipos
df['Data_Inscricao'] = pd.to_datetime(df['Data_Inscricao'], errors='coerce')
df['Data_Finalizacao'] = pd.to_datetime(df['Data_Finalizacao'], errors='coerce')

df.info()
print('-' * 100)
print(df.head(10))
print(f"\nExistem {df.shape[0]} linhas e {df.shape[1]} colunas no arquivo.")

## Dicionário

In [ ]:
dictionary = {
    'ID_Candidato': 'Identificador único do candidato',
    'Status': 'Resultado final do processo seletivo (Aprovado / Reprovado)',
    'Data_Inscricao': 'Data em que o candidato se inscreveu na vaga',
    'Data_Finalizacao': 'Data em que o processo foi encerrado',
    'Motivo_Reprovacao': 'Motivo da reprovação (vazio para aprovados)',
    'Tempo_Processo_Dias': 'Duração do processo em dias',
    'Pretensao_Salarial': 'Pretensão salarial informada (R$)',
    'Recrutador': 'Recrutador responsável pelo processo',
    'Origem_Candidato': 'Canal de origem do candidato',
    'Senioridade': 'Nível de senioridade (Júnior / Pleno / Sênior)',
    'Area': 'Área da vaga',
    'Teste_Tecnico': 'Resultado do teste técnico'
}

categoricas = ['Status', 'Motivo_Reprovacao', 'Recrutador', 'Origem_Candidato',
               'Senioridade', 'Area', 'Teste_Tecnico']
numericas = ['Tempo_Processo_Dias', 'Pretensao_Salarial']
datas = ['Data_Inscricao', 'Data_Finalizacao']

def classificar_tipo(var):
    if var in categoricas:
        return 'Categórica'
    elif var in numericas:
        return 'Numérica'
    elif var in datas:
        return 'Data'
    return 'Outros'  # ex.: ID_Candidato

dict_df = pd.DataFrame(dictionary.items(), columns=['Variável', 'Info'])
dict_df['Tipo'] = dict_df['Variável'].apply(classificar_tipo)
dict_df = dict_df.sort_values(['Tipo', 'Variável']).reset_index(drop=True)
display(dict_df)

## Qualidade dos dados

In [ ]:
print(f"Instâncias: {df.shape[0]} | Atributos: {df.shape[1]}")
print(f"Duplicadas: {df.duplicated().sum()}")
print(f"IDs duplicados: {df['ID_Candidato'].duplicated().sum()}")

missing_pct = df.isnull().mean() * 100
zeros_pct = (df == 0).mean() * 100
miss_zeros = pd.DataFrame({'feat': df.columns, 'miss_%': missing_pct, 'zero_%': zeros_pct}) \
    .sort_values('miss_%', ascending=False)
display(miss_zeros.head(10))

# Motivo_Reprovacao só é preenchido para reprovados: nulo aqui é esperado
print('\nNulos em Motivo_Reprovacao por Status:')
print(df.groupby('Status')['Motivo_Reprovacao'].apply(lambda s: s.isna().mean().round(2)))

In [ ]:
df['target'] = np.where(df['Status'].str.lower() == 'aprovado', 1,
                np.where(df['Status'].str.lower() == 'reprovado', 0, np.nan))

df = df[df['target'].notna()].copy()
df['target'] = df['target'].astype(int)

print('Distribuição Target:')
display(df['Status'].value_counts())
display(df['Status'].value_counts(normalize=True).round(3))

aprovados_df = df[df['target'] == 1]
reprovados_df = df[df['target'] == 0]

## Importância univariada (KS) das variáveis numéricas

In [ ]:
numvars = ['Pretensao_Salarial', 'Tempo_Processo_Dias']
numvars_exist = [c for c in numvars if c in df.columns]
for c in numvars_exist:
    df[c] = pd.to_numeric(df[c], errors='coerce')

def ks_feature_importance(X, y):
    ks_scores = []
    for col in X.columns:
        g0 = X.loc[y == 0, col].dropna()
        g1 = X.loc[y == 1, col].dropna()
        if len(g0) > 0 and len(g1) > 0:
            ks_stat, p_value = ks_2samp(g0, g1)
            ks_scores.append({'feature': col, 'ks_stat': ks_stat, 'p_value': p_value})
    if not ks_scores:
        return pd.DataFrame(columns=['feature', 'ks_stat', 'p_value'])
    return pd.DataFrame(ks_scores).sort_values('ks_stat', ascending=False)

ks_df = ks_feature_importance(df[numvars_exist], df['target'])
display(ks_df)

> `Tempo_Processo_Dias` só é conhecido **depois** do desfecho (e nesta base é gerado a partir do status), então um KS alto nele não é um preditor utilizável, é *vazamento de informação*. O mesmo vale para `Motivo_Reprovacao`, `Data_Finalizacao` e, em grande parte, `Teste_Tecnico`.

## Correlação entre variáveis numéricas

In [ ]:
df_corr = df[numvars_exist + ['target']].copy()
display(df_corr.describe())

corr = df_corr.corr()
fig, ax = novo_grafico((8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax,
            annot_kws={'color': 'white'}, cbar=False)
ax.set_title('Correlação entre Variáveis Numéricas', color='white', fontsize=16, weight='bold', pad=15)
ax.tick_params(colors=TXT)
plt.tight_layout()
plt.show()

## Análise exploratória

In [ ]:
def contagem_por_status(col, top_n=None, horizontal=True, titulo=None, figsize=(12, 6)):
    """Contagem de Aprovados x Reprovados por categoria (ordenada pelos aprovados)."""
    ordem = df[df['Status'] == 'Aprovado'][col].value_counts()
    if top_n:
        ordem = ordem.nlargest(top_n)
    ordem = ordem.index
    d = df[df[col].isin(ordem)]

    fig, ax = novo_grafico(figsize)
    kw = dict(y=col) if horizontal else dict(x=col)
    sns.countplot(data=d, hue='Status', hue_order=HUE_ORDER, order=ordem,
                  palette=PALETA, ax=ax, **kw)
    estilizar(ax, titulo or f'{col} por Status (Aprovados vs Reprovados)',
              xlabel='Quantidade' if horizontal else col,
              ylabel=col if horizontal else 'Quantidade',
              grid_eixo='x' if horizontal else 'y')
    rotular_barras(ax, horizontal)
    if not horizontal:
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Origem do candidato
contagem_por_status('Origem_Candidato', titulo='Origem do Candidato por Status')

In [ ]:
# Senioridade e Área
contagem_por_status('Senioridade', horizontal=False, figsize=(10, 6))
contagem_por_status('Area', horizontal=False, figsize=(12, 6))

In [ ]:
# Taxa de aprovação por categoria
def taxa_aprovacao(col, figsize=(10, 5)):
    t = df.groupby(col)['target'].mean().sort_values(ascending=False) * 100
    fig, ax = novo_grafico(figsize)
    ax.bar(t.index, t.values, color=AZUL)
    ax.axhline(df['target'].mean() * 100, color=VERMELHO, linestyle='--', label='Média geral')
    ax.legend()
    estilizar(ax, f'Taxa de aprovação por {col}', col, '% aprovados')
    for i, v in enumerate(t.values):
        ax.text(i, v + 0.5, f'{v:.0f}%', ha='center', color='white', fontsize=10)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

for c in ['Origem_Candidato', 'Area', 'Senioridade', 'Recrutador']:
    taxa_aprovacao(c)

In [ ]:
# Top motivos de reprovação
reprovados = df[df['Status'] == 'Reprovado']
motivos_cont = reprovados['Motivo_Reprovacao'].value_counts().nlargest(10)

fig, ax = novo_grafico((12, 6))
bars = ax.barh(motivos_cont.index, motivos_cont.values, color=VERMELHO)
ax.invert_yaxis()
estilizar(ax, 'Principais Motivos de Reprovação', 'Quantidade', 'Motivo', grid_eixo='x')
rotular_barras(ax, horizontal=True)
plt.tight_layout()
plt.show()

In [ ]:
# Motivos de reprovação por Senioridade (proporção)
ct = pd.crosstab(reprovados['Senioridade'], reprovados['Motivo_Reprovacao'], normalize='index')

fig, ax = novo_grafico((14, 6))
ct.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Motivo')
estilizar(ax, 'Motivos de Reprovação por Senioridade', 'Senioridade', 'Proporção')
plt.xticks(rotation=0)
plt.subplots_adjust(right=0.78)
plt.show()

In [ ]:
# Tempo do processo: Aprovado x Reprovado
fig, ax = novo_grafico((10, 6))
sns.boxplot(data=df, x='Status', y='Tempo_Processo_Dias', order=HUE_ORDER,
            palette=PALETA, hue='Status', legend=False, ax=ax)
estilizar(ax, 'Tempo do processo: Aprovado x Reprovado', '', 'Tempo do processo (dias)')
plt.tight_layout()
plt.show()

# Pretensão salarial por Senioridade e Status
fig, ax = novo_grafico((12, 6))
sns.boxplot(data=df, x='Senioridade', y='Pretensao_Salarial',
            order=['Júnior', 'Pleno', 'Sênior'], hue='Status', hue_order=HUE_ORDER,
            palette=PALETA, ax=ax)
estilizar(ax, 'Pretensão Salarial por Senioridade e Status', 'Senioridade', 'Pretensão salarial (R$)')
plt.tight_layout()
plt.show()

In [ ]:
# Teste técnico x Status
ct_teste = pd.crosstab(df['Teste_Tecnico'], df['Status'])
display(ct_teste)

fig, ax = novo_grafico((8, 5))
sns.countplot(data=df, x='Teste_Tecnico', hue='Status', hue_order=HUE_ORDER, palette=PALETA, ax=ax)
estilizar(ax, 'Teste Técnico por Status', 'Resultado do teste', 'Quantidade')
rotular_barras(ax)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução mensal por data de inscrição (todos os meses da base)
df['Mes_Ano'] = df['Data_Inscricao'].dt.to_period('M').astype(str)
ordem_meses = sorted(df['Mes_Ano'].dropna().unique())

fig, ax = novo_grafico((12, 6))
sns.countplot(data=df, x='Mes_Ano', hue='Status', hue_order=HUE_ORDER,
              order=ordem_meses, palette=PALETA, ax=ax)
estilizar(ax, 'Candidatos Aprovados x Reprovados por Mês de Inscrição', 'Mês/Ano', 'Quantidade')
rotular_barras(ax)
plt.tight_layout()
plt.show()

# Comparativo YoY como no original não se aplica: a base cobre ~6 meses (out/2025 a abr/2026).

## Clusterização (K-Means) por Pretensão Salarial e Tempo de Processo

In [ ]:
cols = ['Pretensao_Salarial', 'Tempo_Processo_Dias']
df_model_km = df[cols].dropna().reset_index(drop=True)

scaler = StandardScaler()
X_km = scaler.fit_transform(df_model_km)

sse, silhouettes = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    sse.append(km.inertia_)
    silhouettes.append(silhouette_score(X_km, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(list(K_range), sse, marker='o')
ax[0].set_title('Método Elbow'); ax[0].set_xlabel('Número de clusters'); ax[0].set_ylabel('SSE')
ax[1].plot(list(K_range), silhouettes, marker='o', color='green')
ax[1].set_title('Método Silhouette'); ax[1].set_xlabel('Número de clusters'); ax[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(silhouettes))]
print('Melhor K sugerido:', best_k)

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_model_km['cluster'] = kmeans.fit_predict(X_km)

# Traz o Status de volta (mesmo índice, pois dropna não removeu linhas se não houver nulos)
df_km_full = df.dropna(subset=cols).reset_index(drop=True).copy()
df_km_full['cluster'] = df_model_km['cluster']

print('Resumo dos clusters:')
display(df_model_km.groupby('cluster')[cols].agg(['count', 'mean', 'median', 'min', 'max']).round(2))

print('\nPercentual por cluster:')
print((df_model_km['cluster'].value_counts(normalize=True).sort_index() * 100).round(2))

centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=cols)
centroids['cluster'] = range(best_k)
print('\nCentróides na escala original:')
display(centroids.round(2))

print('\nTaxa de aprovação por cluster:')
display((df_km_full.groupby('cluster')['target'].mean() * 100).round(1).rename('% aprovados'))
display(pd.crosstab(df_km_full['cluster'], df_km_full['Status']))

In [ ]:
fig, ax = novo_grafico((10, 6))
sns.scatterplot(data=df_model_km, x='Tempo_Processo_Dias', y='Pretensao_Salarial',
                hue='cluster', palette='tab10', s=70, alpha=0.7, ax=ax)
ax.scatter(centroids['Tempo_Processo_Dias'], centroids['Pretensao_Salarial'],
           c='white', s=250, marker='X', label='Centróides')
ax.legend()
estilizar(ax, 'Clusters K-Means com Centróides', 'Tempo do processo (dias)', 'Pretensão salarial (R$)')
plt.tight_layout()
plt.show()

fig, ax = novo_grafico((8, 5))
sns.countplot(data=df_model_km, x='cluster', hue='cluster', palette='tab10', legend=False, ax=ax)
estilizar(ax, 'Quantidade de Registros por Cluster', 'Cluster', 'Quantidade')
rotular_barras(ax)
plt.tight_layout()
plt.show()

for var in cols:
    fig, ax = novo_grafico((8, 5))
    sns.boxplot(data=df_model_km, x='cluster', y=var, hue='cluster', palette='tab10', legend=False, ax=ax)
    estilizar(ax, f'Distribuição de {var} por Cluster', 'Cluster', var)
    plt.tight_layout()
    plt.show()

## Modelo preditivo (Árvore de Decisão) + SHAP

**Escolha das features:** usei apenas variáveis conhecidas **no momento da inscrição/triagem** (pretensão, senioridade, área, recrutador, origem). Ficaram de fora, por serem vazamento de informação, `Tempo_Processo_Dias`, `Data_Finalizacao`, `Motivo_Reprovacao` (só existe para reprovados) e `Teste_Tecnico` (todo aprovado passou no teste).

> Como esta base é **fictícia e gerada aleatoriamente** (exceto o salário por senioridade), é esperado que o modelo tenha desempenho próximo ao acaso (AUC ≈ 0,5). Se incluir as variáveis de vazamento, o AUC sobe artificialmente.

In [ ]:
# 1) FEATURES
candidate_features = ['Pretensao_Salarial', 'Senioridade', 'Area', 'Recrutador', 'Origem_Candidato']
# Para ver o efeito do vazamento (apenas demonstração), acrescente:
# candidate_features += ['Tempo_Processo_Dias', 'Teste_Tecnico']

features = [c for c in candidate_features if c in df.columns]

df_m = df.dropna(subset=['Data_Inscricao']).sort_values('Data_Inscricao').reset_index(drop=True)
X = df_m[features].copy()
y = df_m['target'].copy()

numeric_features = [c for c in features if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in features if c not in numeric_features]

# 2) PRÉ-PROCESSAMENTO
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 3) SPLIT TEMPORAL (70% mais antigos para treino, 30% mais recentes para teste)
split_index = int(len(df_m) * 0.7)
X_train, y_train = X.iloc[:split_index], y.iloc[:split_index]
X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]

print('Treino:', X_train.shape, '| período:', df_m['Data_Inscricao'].iloc[0].date(),
      'a', df_m['Data_Inscricao'].iloc[split_index - 1].date())
print('Teste :', X_test.shape, '| período:', df_m['Data_Inscricao'].iloc[split_index].date(),
      'a', df_m['Data_Inscricao'].iloc[-1].date())

# 4) MODELO
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=3, min_samples_split=50, min_samples_leaf=30,
        class_weight='balanced', random_state=42))
])
model.fit(X_train, y_train)

# 5) PREDIÇÕES
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 6) NOMES DAS FEATURES
feature_names = [n.split('__', 1)[1] for n in model.named_steps['preprocessor'].get_feature_names_out()]
tree_clf = model.named_steps['classifier']

In [ ]:
# FEATURE IMPORTANCE
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': tree_clf.feature_importances_}) \
    .sort_values('importance', ascending=False)
print('=== FEATURE IMPORTANCE ===')
print(feat_imp.head(10))

fig, ax = novo_grafico((10, 6))
sns.barplot(data=feat_imp.head(10), x='importance', y='feature', color=AZUL, ax=ax)
estilizar(ax, 'Feature Importance', 'Importância', '', grid_eixo='x')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP
X_test_t = model.named_steps['preprocessor'].transform(X_test).astype(float)

explainer = shap.TreeExplainer(tree_clf)
sv = explainer.shap_values(X_test_t)

# Compatível com diferentes versões do shap
if isinstance(sv, list):
    sv_pos = sv[1]
elif getattr(sv, 'ndim', 2) == 3:
    sv_pos = sv[:, :, 1]
else:
    sv_pos = sv

shap.summary_plot(sv_pos, X_test_t, feature_names=feature_names)

In [ ]:
# MÉTRICAS
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)

print('=== MÉTRICAS DO MODELO ===')
print(f'Acurácia : {acc:.4f}')
print(f'Precisão : {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'AUC      : {auc:.4f}')

print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['Reprovado', 'Aprovado'], zero_division=0))

In [ ]:
# MATRIZ DE CONFUSÃO
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred. Reprovado', 'Pred. Aprovado'],
            yticklabels=['Real Reprovado', 'Real Aprovado'])
plt.title('Matriz de Confusão')
plt.xlabel('Predição'); plt.ylabel('Real')
plt.tight_layout()
plt.show()

# CURVA ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.show()

# ÁRVORE
plt.figure(figsize=(22, 10))
plot_tree(tree_clf, feature_names=feature_names, class_names=['Reprovado', 'Aprovado'],
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree')
plt.show()